In [1]:
%pip install pandas requests beautifulsoup4 playwright nest_asyncio

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ------- -------------------------------- 2.1/10.7 MB 13.2 MB/s eta 0:00:01
   -------------------- ------------------- 5.5/10.7 MB 14.5 MB/s eta 0:00:01
   ------------------------------- -------- 8.4/10.7 MB 14.1 MB/s eta 0:00:01
   ----------------------------------- ---- 9.4/10.7 MB 12.5 MB/s eta 0:00:01
   ---------------------------------------- 10.7/10.7 MB 11.0 MB/s eta 0:00:00
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   - -------------------------------------- 1.8/38.8 MB 8.5 MB/s eta 0:00:05
   --- ------------------------------------ 3.7/38.8 MB 8.6 MB/s eta 0:00:05
   ----- ---------------------------------- 5.5/38.8 MB 8.8 MB/s eta 0:00:04
   ------- -------------------------------- 6.8/3


[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import asyncio
import nest_asyncio
import pandas as pd
import requests
import re
import os
import time

from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urljoin

nest_asyncio.apply()

print("Library berhasil diimpor!")

Library berhasil diimpor!


In [3]:
BASE_URL = "https://putusan3.mahkamahagung.go.id"

URL_TAHUN = {
    2023: f"{BASE_URL}/direktori/index/pengadilan/pn-surabaya/kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/2023.html",

    2024: f"{BASE_URL}/direktori/index/pengadilan/pn-surabaya/kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/2024.html",

    2025: f"{BASE_URL}/direktori/index/pengadilan/pn-surabaya/kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/2025.html"
}

TARGET_PDF = 50

FOLDER_DATA = Path("data")
FOLDER_PDF = Path("PDF_Putusan")
FOLDER_TXT = Path("TXT_Putusan")

for folder in [
    FOLDER_DATA,
    FOLDER_PDF / "2023",
    FOLDER_PDF / "2024",
    FOLDER_PDF / "2025",
    FOLDER_TXT / "2023",
    FOLDER_TXT / "2024",
    FOLDER_TXT / "2025"
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Konfigurasi selesai!")

Konfigurasi selesai!


In [4]:
for tahun, url in URL_TAHUN.items():

    print("=" * 60)
    print("TAHUN:", tahun)

    try:
        response = requests.get(
            url,
            timeout=30,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        print("HTTP:", response.status_code)

        if response.status_code == 200:
            soup = BeautifulSoup(
                response.text,
                "html.parser"
            )

            print("Judul:", soup.title.text.strip())
            print("Halaman berhasil dibaca.")

        else:
            print("Akses belum berhasil.")

    except Exception as e:
        print("ERROR:", e)

TAHUN: 2023
HTTP: 403
Akses belum berhasil.
TAHUN: 2024
HTTP: 403
Akses belum berhasil.
TAHUN: 2025
HTTP: 403
Akses belum berhasil.


In [5]:

from playwright.async_api import async_playwright

async def test_browser_access():

    async with async_playwright() as p:

        browser = await p.chromium.connect_over_cdp(
            "http://127.0.0.1:9222"
        )

        context = browser.contexts[0]

        page = (
            context.pages[0]
            if context.pages
            else await context.new_page()
        )

        url = URL_TAHUN[2023]

        print("Menguji akses halaman tahun 2023...")
        
        response = await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000
        )

        print("HTTP:", response.status if response else None)
        print("URL akhir:", page.url)
        print("Judul:", await page.title())

        text = await page.locator("body").inner_text()
        print("\nCUPLIKAN HALAMAN:")
        print(text[:2000])

        await browser.close()

await test_browser_access()


Menguji akses halaman tahun 2023...
HTTP: 200
URL akhir: https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-surabaya/kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/2023.html
Judul: Direktori Putusan

CUPLIKAN HALAMAN:
BERANDA
PENCARIAN
DIREKTORI
PENGADILAN
PERATURAN
TENTANG
Beranda
Pengadilan
PN SURABAYA
Pidana Khusus
Narkotika dan Psikotropika
Putus
2023
PUTUSAN PN SURABAYA
NARKOTIKA DAN PSIKOTROPIKA
PUTUS TAHUN 2023
Direktori

Semua Direktori

Perdata 4202

Pidana Umum 2492

Pidana Khusus 1538

Perdata Agama 526

Perdata Khusus 345

Pidana Militer 1

Klasifikasi

Narkotika dan Psikotropika 1054

Pengadilan

PN SURABAYA 1054

Tahun
Putus	Register	Upload


2026 460  


2025 822  


2024 715  


2023 1054  


2022 1185  


selengkapnya 	

2026 324  


2025 761  


2024 814  


2023 930  


2022 1242  


selengkapnya 	

2026 803  


2025 759  


2024 654  


2023 1123  


2022 1182  


selengkapnya 
Bulan

Januari 136

Februari 102

Maret 93

April 59

Mei 9

In [6]:
async def inspect_decision_links():

    async with async_playwright() as p:

        browser = await p.chromium.connect_over_cdp(
            "http://127.0.0.1:9222"
        )

        context = browser.contexts[0]
        page = context.pages[0]

        await page.goto(
            URL_TAHUN[2023],
            wait_until="domcontentloaded"
        )

        links = await page.locator("a[href]").evaluate_all("""
            elements => elements.map(a => ({
                text: (a.innerText || a.textContent || '').trim(),
                href: a.href
            })).filter(a =>
                a.href.includes('/direktori/putusan/')
            )
        """)

        print("Total link putusan:", len(links))

        for link in links[:10]:
            print("\\nJudul:", link["text"])
            print("URL:", link["href"])

        await browser.close()

await inspect_decision_links()

Total link putusan: 43
\nJudul: Putusan PN SURABAYA Nomor 1512/Pid.Sus/2023/PN Sby
URL: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaeec95fd2b30f568e96313133333038.html
\nJudul: Putusan PN SURABAYA Nomor 2256/Pid.Sus/2023/PN Sby
URL: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaeeb3b51b0ea8349cc2323134383131.html
\nJudul: Putusan PN SURABAYA Nomor 2257/Pid.Sus/2023/PN Sby
URL: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaeeb3b50c0e21debfde323134373436.html
\nJudul: Putusan PN SURABAYA Nomor 2389/Pid.Sus/2023/PN Sby
URL: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaeeb3b4b16da786b453323134353134.html
\nJudul: Putusan PN SURABAYA Nomor 2282/Pid.Sus/2023/PN Sby
URL: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaeeb3b4954d60968798323134343237.html
\nJudul: Putusan PN SURABAYA Nomor 2395/Pid.Sus/2023/PN Sby
URL: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaeea0695601b9e0b718303832373537.html
\nJudul: Putusan PN SURABAYA 

In [7]:
async def inspect_pagination():

    async with async_playwright() as p:

        browser = await p.chromium.connect_over_cdp(
            "http://127.0.0.1:9222"
        )

        page = browser.contexts[0].pages[0]

        await page.goto(
            URL_TAHUN[2023],
            wait_until="domcontentloaded"
        )

        # Ambil seluruh link putusan
        decision_links = await page.locator("a[href]").evaluate_all("""
            elements => elements
                .filter(a => a.href.includes('/direktori/putusan/'))
                .map(a => ({
                    title: (a.innerText || a.textContent || '').trim(),
                    url: a.href
                }))
        """)

        unique_links = {
            item["url"]: item for item in decision_links
        }

        print("Total link:", len(decision_links))
        print("Link unik:", len(unique_links))

        # Periksa link pagination
        pagination = await page.locator("a[href]").evaluate_all("""
            elements => elements.map(a => ({
                text: (a.innerText || a.textContent || '').trim(),
                href: a.href
            })).filter(a =>
                /page\\//i.test(a.href)
                || /berikutnya|next|selanjutnya/i.test(a.text)
            )
        """)

        print("\\nLINK PAGINATION:")

        for item in pagination:
            print(item)

        await browser.close()


await inspect_pagination()

Total link: 43
Link unik: 37
\nLINK PAGINATION:
{'text': '2', 'href': 'https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-surabaya/kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/2023/page/2.html'}
{'text': '3', 'href': 'https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-surabaya/kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/2023/page/3.html'}
{'text': '4', 'href': 'https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-surabaya/kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/2023/page/4.html'}
{'text': 'Next', 'href': 'https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-surabaya/kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/2023/page/2.html'}
{'text': 'Last', 'href': 'https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-surabaya/kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/2023/page/53.html'}


In [8]:

async def test_page_2():

    async with async_playwright() as p:

        browser = await p.chromium.connect_over_cdp(
            "http://127.0.0.1:9222"
        )

        context = browser.contexts[0]
        page = context.pages[0]

        url = (
            "https://putusan3.mahkamahagung.go.id/"
            "direktori/index/pengadilan/pn-surabaya/"
            "kategori/narkotika-dan-psikotropika-1/"
            "tahunjenis/putus/tahun/2023/page/2.html"
        )

        print("Menguji halaman 2 tahun 2023...")

        response = await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000
        )

        print("HTTP:", response.status if response else None)
        print("URL akhir:", page.url)
        print("Judul:", await page.title())

        links = await page.locator("a[href]").evaluate_all("""
            elements => elements
                .filter(a => a.href.includes('/direktori/putusan/'))
                .map(a => ({
                    title: (a.innerText || a.textContent || '').trim(),
                    url: a.href
                }))
        """)

        unique = {
            item["url"]: item for item in links
        }

        print("Total link:", len(links))
        print("Link unik:", len(unique))

        for item in list(unique.values())[:5]:
            print(item["title"])
            print(item["url"])

        await browser.close()


await test_page_2()


Menguji halaman 2 tahun 2023...
HTTP: 200
URL akhir: https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-surabaya/kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/2023/page/2.html
Judul: Direktori Putusan
Total link: 43
Link unik: 42
Putusan PN SURABAYA Nomor 2219/Pid.Sus/2023/PN Sby
https://putusan3.mahkamahagung.go.id/direktori/putusan/zaee9d96f3af8a58ae24313831363535.html
Putusan PN SURABAYA Nomor 2417/Pid.Sus/2023/PN Sby
https://putusan3.mahkamahagung.go.id/direktori/putusan/zaee9d805ac7e562ad8e313533353039.html
Putusan PN SURABAYA Nomor 2258/Pid.Sus/2023/PN Sby
https://putusan3.mahkamahagung.go.id/direktori/putusan/zaee9d7fc537bb9ebb33313533303539.html
Putusan PN SURABAYA Nomor 1918/Pid.Sus/2023/PN Sby
https://putusan3.mahkamahagung.go.id/direktori/putusan/zaee9a726f86f36eba5a313831373538.html
Putusan PN SURABAYA Nomor 2090/Pid.Sus/2023/PN Sby
https://putusan3.mahkamahagung.go.id/direktori/putusan/zaee9a6cd224ba0cbdb8313733373436.html


In [10]:

import csv
import re
import asyncio
import pandas as pd
from pathlib import Path
from playwright.async_api import async_playwright

# ==========================================
# KONFIGURASI
# ==========================================

MAX_HALAMAN = 5
DELAY = 5

OUTPUT = Path("data/metadata_putusan.csv")
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://putusan3.mahkamahagung.go.id"

URL_TAHUN = {
    2023: f"{BASE_URL}/direktori/index/pengadilan/pn-surabaya/kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/2023.html",
    2024: f"{BASE_URL}/direktori/index/pengadilan/pn-surabaya/kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/2024.html",
    2025: f"{BASE_URL}/direktori/index/pengadilan/pn-surabaya/kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/2025.html"
}

# ==========================================
# FUNGSI EKSTRAKSI LINK
# ==========================================


async def extract_metadata(page, tahun):

    links = await page.locator("a[href]").evaluate_all("""
        elements => elements.map(a => ({
            title: (a.innerText || a.textContent || '').trim(),
            url: a.href
        })).filter(a =>
            a.url.includes('/direktori/putusan/')
        )
    """)

    results = []

    for item in links:

        title = re.sub(r"\s+", " ", item["title"]).strip()

        match = re.search(
            r"Nomor\s+(.+)$",
            title,
            re.IGNORECASE
        )

        if not match:
            continue

        nomor = match.group(1).strip()

        results.append({
            "No Putusan": nomor,
            "Tahun Filter": tahun,
            "Lembaga Peradilan": "Pengadilan Negeri Surabaya",
            "Judul Putusan": title,
            "URL Detail": item["url"]
        })

    return results



# ==========================================
# FUNGSI SIMPAN PROGRES
# ==========================================

def save_metadata(data):

    df = pd.DataFrame(data)

    if not df.empty:
        df = df.drop_duplicates(
            subset=["URL Detail"]
        )

        df = df.reset_index(drop=True)
        df.insert(0, "No", range(1, len(df) + 1))

    df.to_csv(
        OUTPUT,
        index=False,
        encoding="utf-8-sig"
    )

    return df


# ==========================================
# CRAWLING UTAMA
# ==========================================

async def crawling_metadata():

    all_data = []
    visited_urls = set()

    async with async_playwright() as p:

        browser = await p.chromium.connect_over_cdp(
            "http://127.0.0.1:9222"
        )

        context = browser.contexts[0]

        page = (
            context.pages[0]
            if context.pages
            else await context.new_page()
        )

        stop_all = False

        for tahun, base_url in URL_TAHUN.items():

            if stop_all:
                break

            print("\\n" + "=" * 60)
            print("CRAWLING TAHUN:", tahun)
            print("=" * 60)

            for nomor_halaman in range(1, MAX_HALAMAN + 1):

                if nomor_halaman == 1:
                    url = base_url
                else:
                    url = (
                        base_url.replace(
                            ".html",
                            f"/page/{nomor_halaman}.html"
                        )
                    )

                if url in visited_urls:
                    continue

                visited_urls.add(url)

                print(
                    f"\\nTahun {tahun} | "
                    f"Halaman {nomor_halaman}"
                )

                try:

                    response = await page.goto(
                        url,
                        wait_until="domcontentloaded",
                        timeout=60000
                    )

                    if response is None:
                        print("Tidak ada respons.")
                        continue

                    print("HTTP:", response.status)

                    if response.status in (403, 429):

                        print(
                            "Pembatasan akses terdeteksi. "
                            "Crawler dihentikan."
                        )

                        stop_all = True
                        break

                    if response.status != 200:
                        print("Halaman dilewati.")
                        continue

                    hasil = await extract_metadata(
                        page,
                        tahun
                    )

                    jumlah_sebelum = len(all_data)

                    for item in hasil:
                        if item["URL Detail"] not in {
                            x["URL Detail"] for x in all_data
                        }:
                            all_data.append(item)

                    tambahan = len(all_data) - jumlah_sebelum

                    print("Link ditemukan:", len(hasil))
                    print("Data baru:", tambahan)
                    print("Total unik:", len(all_data))

                    save_metadata(all_data)

                    await asyncio.sleep(DELAY)

                except Exception as error:

                    print("Error:", repr(error))
                    stop_all = True
                    break

        await browser.close()

    # ======================================
    # REKAP
    # ======================================

    df = save_metadata(all_data)

    print("\\n" + "=" * 60)
    print("CRAWLING SELESAI")
    print("=" * 60)

    print("Total metadata unik:", len(df))

    if not df.empty:
        print("\\nJumlah per tahun:")
        print(df["Tahun Filter"].value_counts().sort_index())

    print("\\nFile tersimpan:", OUTPUT.resolve())

    return df


df_metadata = await crawling_metadata()

display(df_metadata.head(10))


\n============================================================
CRAWLING TAHUN: 2023
\nTahun 2023 | Halaman 1
HTTP: 200
Link ditemukan: 43
Data baru: 37
Total unik: 37
\nTahun 2023 | Halaman 2
HTTP: 200
Link ditemukan: 43
Data baru: 22
Total unik: 59
\nTahun 2023 | Halaman 3
HTTP: 200
Link ditemukan: 43
Data baru: 23
Total unik: 82
\nTahun 2023 | Halaman 4
HTTP: 200
Link ditemukan: 43
Data baru: 20
Total unik: 102
\nTahun 2023 | Halaman 5
HTTP: 200
Link ditemukan: 43
Data baru: 20
Total unik: 122
\n============================================================
CRAWLING TAHUN: 2024
\nTahun 2024 | Halaman 1
HTTP: 200
Link ditemukan: 43
Data baru: 43
Total unik: 165
\nTahun 2024 | Halaman 2
HTTP: 200
Link ditemukan: 43
Data baru: 20
Total unik: 185
\nTahun 2024 | Halaman 3
HTTP: 200
Link ditemukan: 43
Data baru: 20
Total unik: 205
\nTahun 2024 | Halaman 4
HTTP: 200
Link ditemukan: 43
Data baru: 18
Total unik: 223
\nTahun 2024 | Halaman 5
HTTP: 200
Link ditemukan: 43
Data baru: 19
Total unik:

,No,No Putusan,Tahun Filter,Lembaga Peradilan,Judul Putusan,URL Detail
0,1,1512/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 1512/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
1,2,2256/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2256/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
2,3,2257/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2257/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
3,4,2389/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2389/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
4,5,2282/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2282/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
5,6,2395/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2395/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
6,7,2216/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2216/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
7,8,2052/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2052/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
8,9,2317/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2317/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
9,10,2053/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2053/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...


In [11]:
df_metadata = pd.read_csv("data/metadata_putusan.csv")

print("Total data:", len(df_metadata))

print("\nDistribusi tahun:")
print(df_metadata["Tahun Filter"].value_counts().sort_index())

display(df_metadata.head(10))

Total data: 362

Distribusi tahun:
Tahun Filter
2023    122
2024    120
2025    120
Name: count, dtype: int64


,No,No Putusan,Tahun Filter,Lembaga Peradilan,Judul Putusan,URL Detail
0,1,1512/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 1512/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
1,2,2256/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2256/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
2,3,2257/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2257/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
3,4,2389/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2389/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
4,5,2282/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2282/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
5,6,2395/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2395/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
6,7,2216/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2216/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
7,8,2052/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2052/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
8,9,2317/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2317/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...
9,10,2053/Pid.Sus/2023/PN Sby,2023,Pengadilan Negeri Surabaya,Putusan PN SURABAYA Nomor 2053/Pid.Sus/2023/PN...,https://putusan3.mahkamahagung.go.id/direktori...


AUDIT PDF

In [12]:

import asyncio
import pandas as pd
from pathlib import Path
from playwright.async_api import async_playwright

INPUT = Path("data/metadata_putusan.csv")
OUTPUT = Path("data/hasil_audit_pdf.csv")

df = pd.read_csv(INPUT)

async def audit_pdf():

    hasil = []

    async with async_playwright() as p:

        browser = await p.chromium.connect_over_cdp(
            "http://127.0.0.1:9222"
        )

        context = browser.contexts[0]
        page = context.pages[0]

        for i, row in df.iterrows():

            nomor = row["No Putusan"]
            url = row["URL Detail"]

            print(
                f"\n[{i+1}/{len(df)}] {nomor}"
            )

            status_pdf = ""
            url_pdf = ""
            keterangan = ""

            try:

                response = await page.goto(
                    url,
                    wait_until="domcontentloaded",
                    timeout=60000
                )

                if response is None:
                    status_pdf = "ERROR_HALAMAN"
                    keterangan = "Tidak ada respons"

                elif response.status in (403, 429):
                    status_pdf = "AKSES_DITOLAK"
                    keterangan = f"HTTP {response.status}"

                elif response.status != 200:
                    status_pdf = "ERROR_HALAMAN"
                    keterangan = f"HTTP {response.status}"

                else:

                    links = await page.locator(
                        "a[href]"
                    ).evaluate_all("""
                        elements => elements.map(a => ({
                            text: (a.innerText || a.textContent || '').trim(),
                            href: a.href
                        }))
                    """)

                    pdf_links = [
                        x for x in links
                        if "/pdf/" in x["href"].lower()
                        and "/zip/" not in x["href"].lower()
                    ]

                    if pdf_links:
                        status_pdf = "PDF_TERSEDIA"
                        url_pdf = pdf_links[0]["href"]
                        keterangan = "Tautan PDF ditemukan"

                    else:
                        status_pdf = "TIDAK_ADA_PDF"
                        keterangan = "Tautan PDF tidak ditemukan"

            except Exception as e:
                status_pdf = "ERROR_HALAMAN"
                keterangan = str(e)[:200]

            hasil.append({
                **row.to_dict(),
                "URL PDF": url_pdf,
                "Status PDF": status_pdf,
                "Keterangan": keterangan
            })

            # Simpan progres setiap putusan.
            pd.DataFrame(hasil).to_csv(
                OUTPUT,
                index=False,
                encoding="utf-8-sig"
            )

            print("Status:", status_pdf)

            # Berhenti jika server membatasi akses.
            if status_pdf == "AKSES_DITOLAK":
                print("Akses dibatasi. Audit dihentikan.")
                break

            await asyncio.sleep(3)

        await browser.close()

    return pd.DataFrame(hasil)


df_audit = await audit_pdf()



[1/362] 1512/Pid.Sus/2023/PN Sby
Status: PDF_TERSEDIA

[2/362] 2256/Pid.Sus/2023/PN Sby
Status: TIDAK_ADA_PDF

[3/362] 2257/Pid.Sus/2023/PN Sby
Status: TIDAK_ADA_PDF

[4/362] 2389/Pid.Sus/2023/PN Sby
Status: TIDAK_ADA_PDF

[5/362] 2282/Pid.Sus/2023/PN Sby
Status: TIDAK_ADA_PDF

[6/362] 2395/Pid.Sus/2023/PN Sby
Status: TIDAK_ADA_PDF

[7/362] 2216/Pid.Sus/2023/PN Sby
Status: TIDAK_ADA_PDF

[8/362] 2052/Pid.Sus/2023/PN Sby
Status: TIDAK_ADA_PDF

[9/362] 2317/Pid.Sus/2023/PN Sby
Status: TIDAK_ADA_PDF

[10/362] 2053/Pid.Sus/2023/PN Sby
Status: TIDAK_ADA_PDF

[11/362] 1948/Pid.Sus/2023/PN Sby
Status: PDF_TERSEDIA

[12/362] 2129/Pid.Sus/2023/PN Sby
Status: TIDAK_ADA_PDF

[13/362] 1997/Pid.Sus/2023/PN Sby
Status: TIDAK_ADA_PDF

[14/362] 2438/Pid.Sus/2023/PN Sby
Status: PDF_TERSEDIA

[15/362] 2255/Pid.Sus/2023/PN Sby
Status: TIDAK_ADA_PDF

[16/362] 2330/Pid.Sus/2023/PN Sby
Status: PDF_TERSEDIA

[17/362] 2325/Pid.Sus/2023/PN Sby
Status: PDF_TERSEDIA

[18/362] 2327/Pid.Sus/2023/PN Sby
Status: PD

In [13]:
import pandas as pd

# Membaca hasil audit lengkap
df_audit = pd.read_csv("data/hasil_audit_pdf.csv")

print("TOTAL DATA DIAUDIT:", len(df_audit))

print("\nREKAP STATUS PDF:")
print(df_audit["Status PDF"].value_counts())

print("\nREKAP BERDASARKAN TAHUN:")
display(
    pd.crosstab(
        df_audit["Tahun Filter"],
        df_audit["Status PDF"]
    )
)

TOTAL DATA DIAUDIT: 362

REKAP STATUS PDF:
Status PDF
PDF_TERSEDIA     234
TIDAK_ADA_PDF    119
ERROR_HALAMAN      9
Name: count, dtype: int64

REKAP BERDASARKAN TAHUN:


Status PDF,ERROR_HALAMAN,PDF_TERSEDIA,TIDAK_ADA_PDF
Tahun Filter,,,
2023,5,87,30
2024,3,90,27
2025,1,57,62


In [16]:
df_kandidat = df_audit[
    (df_audit["Status PDF"] == "PDF_TERSEDIA") &
    (df_audit["URL PDF"].notna())
].copy()

print("Total kandidat PDF:", len(df_kandidat))

Total kandidat PDF: 234


In [17]:
df_kandidat.to_csv(
    "data/kandidat_pdf.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Daftar kandidat berhasil disimpan!")

Daftar kandidat berhasil disimpan!


SELEKSI 50 PUTUSAN

In [18]:
# Membaca daftar kandidat PDF
df_kandidat = pd.read_csv("data/kandidat_pdf.csv")

# Memastikan hanya PDF yang memiliki URL
df_kandidat = df_kandidat[
    (df_kandidat["Status PDF"] == "PDF_TERSEDIA") &
    (df_kandidat["URL PDF"].notna())
].copy()

# Seleksi berdasarkan tahun
sampel_2023 = df_kandidat[
    df_kandidat["Tahun Filter"] == 2023
].sample(n=17, random_state=42)

sampel_2024 = df_kandidat[
    df_kandidat["Tahun Filter"] == 2024
].sample(n=17, random_state=42)

sampel_2025 = df_kandidat[
    df_kandidat["Tahun Filter"] == 2025
].sample(n=16, random_state=42)

# Gabungkan seluruh sampel
df_50 = pd.concat(
    [sampel_2023, sampel_2024, sampel_2025],
    ignore_index=True
)

# Simpan daftar 50 putusan
df_50.to_csv(
    "data/kandidat_50_putusan.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Total kandidat:", len(df_50))
print("\nDistribusi tahun:")
print(df_50["Tahun Filter"].value_counts().sort_index())

display(
    df_50[["No Putusan", "Tahun Filter", "URL PDF"]]
)

Total kandidat: 50

Distribusi tahun:
Tahun Filter
2023    17
2024    17
2025    16
Name: count, dtype: int64


,No Putusan,Tahun Filter,URL PDF
0,2031/Pid.Sus/2023/PN Sby,2023,https://putusan3.mahkamahagung.go.id/direktori...
1,1512/Pid.Sus/2023/PN Sby,2023,https://putusan3.mahkamahagung.go.id/direktori...
2,2260/Pid.Sus/2023/PN Sby,2023,https://putusan3.mahkamahagung.go.id/direktori...
3,2166/Pid.Sus/2023/PN Sby,2023,https://putusan3.mahkamahagung.go.id/direktori...
4,270/Pid.Sus/2023/PN Sby,2023,https://putusan3.mahkamahagung.go.id/direktori...
5,1982/Pid.Sus/2023/PN Sby,2023,https://putusan3.mahkamahagung.go.id/direktori...
6,140/Pid.Sus/2023/PN Sby,2023,https://putusan3.mahkamahagung.go.id/direktori...
7,2258/Pid.Sus/2023/PN Sby,2023,https://putusan3.mahkamahagung.go.id/direktori...
8,2325/Pid.Sus/2023/PN Sby,2023,https://putusan3.mahkamahagung.go.id/direktori...
9,2043/Pid.Sus/2023/PN Sby,2023,https://putusan3.mahkamahagung.go.id/direktori...


In [19]:
from playwright.async_api import async_playwright
from pathlib import Path
import pandas as pd
import asyncio
import os

# Membaca 50 kandidat
df_50 = pd.read_csv("data/kandidat_50_putusan.csv")

# Folder penyimpanan PDF
FOLDER_PDF = Path("PDF_Putusan")
FOLDER_PDF.mkdir(exist_ok=True)

print("Jumlah kandidat:", len(df_50))

Jumlah kandidat: 50


In [20]:
async def test_download_pdf():

    async with async_playwright() as p:

        browser = await p.chromium.connect_over_cdp(
            "http://127.0.0.1:9222"
        )

        context = browser.contexts[0]
        page = await context.new_page()

        hasil = []

        # Uji 5 kandidat pertama
        for i, row in df_50.head(5).iterrows():

            url_detail = row["URL Detail"]
            no_putusan = str(row["No Putusan"])

            print(f"\n[{i+1}/5] Menguji:", no_putusan)

            try:
                response = await page.goto(
                    url_detail,
                    wait_until="domcontentloaded",
                    timeout=60000
                )

                await page.wait_for_timeout(2000)

                if response is None or response.status != 200:
                    print("Halaman tidak berhasil diakses.")

                    hasil.append({
                        "No Putusan": no_putusan,
                        "Status": "GAGAL_AKSES"
                    })
                    continue

                # Cari tautan PDF aktif
                links = await page.locator("a[href]").evaluate_all("""
                    elements => elements.map(a => ({
                        href: a.href,
                        text: a.innerText
                    }))
                """)

                pdf_links = [
                    item["href"]
                    for item in links
                    if "/pdf/" in item["href"].lower()
                    and "/zip/" not in item["href"].lower()
                ]

                if not pdf_links:
                    print("Tautan PDF tidak ditemukan.")

                    hasil.append({
                        "No Putusan": no_putusan,
                        "Status": "PDF_TIDAK_DITEMUKAN"
                    })
                    continue

                print("Tautan PDF ditemukan:", pdf_links[0])

                hasil.append({
                    "No Putusan": no_putusan,
                    "Status": "LINK_PDF_DITEMUKAN",
                    "URL PDF": pdf_links[0]
                })

                await asyncio.sleep(3)

            except Exception as e:

                print("Error:", e)

                hasil.append({
                    "No Putusan": no_putusan,
                    "Status": "ERROR"
                })

        pd.DataFrame(hasil).to_csv(
            "data/hasil_uji_download.csv",
            index=False,
            encoding="utf-8-sig"
        )

        await page.close()

        print("\nPengujian selesai.")
        print(pd.DataFrame(hasil)["Status"].value_counts())


await test_download_pdf()


[1/5] Menguji: 2031/Pid.Sus/2023/PN Sby
Tautan PDF ditemukan: https://putusan3.mahkamahagung.go.id/direktori/download_file/973f154e3030ff8f89e30a706f6cd782/pdf/zaee8cec5a5e41babedf313331353235

[2/5] Menguji: 1512/Pid.Sus/2023/PN Sby
Tautan PDF ditemukan: https://putusan3.mahkamahagung.go.id/direktori/download_file/11eec962a67aa63a8ac2313135333233/pdf/zaeec95fd2b30f568e96313133333038

[3/5] Menguji: 2260/Pid.Sus/2023/PN Sby
Tautan PDF ditemukan: https://putusan3.mahkamahagung.go.id/direktori/download_file/a23f16004d4269086e7239d6e52bbb3b/pdf/zaee9a5d40c085f0acdc313534363230

[4/5] Menguji: 2166/Pid.Sus/2023/PN Sby
Tautan PDF ditemukan: https://putusan3.mahkamahagung.go.id/direktori/download_file/160f6f9feb1c4cf5d27a38063ef3ae41/pdf/zaee9a5ebe27b95eb03b313535373030

[5/5] Menguji: 270/Pid.Sus/2023/PN Sby
Tautan PDF ditemukan: https://putusan3.mahkamahagung.go.id/direktori/download_file/c7bd6a6fd498bed8e9e7661e10a36fdd/pdf/zaedb8d6c26339f482bd313534363434

Pengujian selesai.
Status
LINK

In [21]:
async def download_satu_pdf():

    async with async_playwright() as p:

        browser = await p.chromium.connect_over_cdp(
            "http://127.0.0.1:9222"
        )

        context = browser.contexts[0]
        page = await context.new_page()

        row = df_50.iloc[0]

        no_putusan = str(row["No Putusan"])
        url_detail = row["URL Detail"]

        print("Menguji download:", no_putusan)

        try:
            response = await page.goto(
                url_detail,
                wait_until="domcontentloaded",
                timeout=60000
            )

            if response is None or response.status != 200:
                print("Halaman gagal diakses.")
                return

            await page.wait_for_timeout(2000)

            # Cari tautan PDF aktif
            pdf_links = await page.locator(
                'a[href*="/pdf/"]'
            ).evaluate_all("""
                elements => elements.map(a => a.href)
            """)

            pdf_links = [
                url for url in pdf_links
                if "/zip/" not in url.lower()
            ]

            if not pdf_links:
                print("Tautan PDF tidak ditemukan.")
                return

            pdf_url = pdf_links[0]

            print("URL PDF:", pdf_url)
            print("Memulai download...")

            # Klik tautan PDF dan tangkap file unduhan
            async with page.expect_download(
                timeout=60000
            ) as download_info:

                await page.evaluate("""
                    (url) => {
                        const link = [...document.querySelectorAll('a')]
                            .find(a => a.href === url);
                        if (link) link.click();
                    }
                """, pdf_url)

            download = await download_info.value

            nama_file = "uji_" + str(download.suggested_filename)

            lokasi_file = FOLDER_PDF / nama_file

            await download.save_as(str(lokasi_file))

            # Validasi file
            with open(lokasi_file, "rb") as f:
                header = f.read(5)

            ukuran = lokasi_file.stat().st_size

            if header == b"%PDF-" and ukuran > 0:
                print("\nBERHASIL! PDF valid.")
                print("Nama file:", nama_file)
                print("Ukuran:", ukuran, "bytes")
            else:
                print("\nFile terunduh tetapi bukan PDF valid.")

        except Exception as e:
            print("\nDownload gagal:", str(e))

        finally:
            await page.close()


await download_satu_pdf()

Menguji download: 2031/Pid.Sus/2023/PN Sby
URL PDF: https://putusan3.mahkamahagung.go.id/direktori/download_file/973f154e3030ff8f89e30a706f6cd782/pdf/zaee8cec5a5e41babedf313331353235
Memulai download...

BERHASIL! PDF valid.
Nama file: uji_putusan_2031_pid.sus_2023_pn_sby_20261030160527.pdf
Ukuran: 147004 bytes


In [22]:
from pathlib import Path

file_pdf = list(Path("PDF_Putusan").glob("*.pdf"))

print("Jumlah PDF:", len(file_pdf))

for file in file_pdf:
    print("Nama:", file.name)
    print("Ukuran:", round(file.stat().st_size / 1024, 2), "KB")

    with open(file, "rb") as f:
        header = f.read(5)

    print("Valid PDF:", header == b"%PDF-")

Jumlah PDF: 1
Nama: uji_putusan_2031_pid.sus_2023_pn_sby_20261030160527.pdf
Ukuran: 143.56 KB
Valid PDF: True
